# XGBoost Waste Forecasting

In [37]:
!pip install optuna


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [38]:
import os
import pickle
import warnings
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

warnings.filterwarnings('ignore')
os.makedirs('models/xgboost_optimized', exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

MEALS = ['breakfast', 'lunch', 'dinner']
TARGET = 'waste_kg'

## Load and Inspect Dataset

In [39]:
df_raw = pd.read_csv('data/waste_features_xgb.csv')
print(f'Raw dataset shape: {df_raw.shape}')
print('Meal values:', df_raw['meal'].unique())

df_raw = df_raw[df_raw['meal'].isin(MEALS)].copy()
print(f'After filtering to breakfast/lunch/dinner: {df_raw.shape}')
df_raw.head()

Raw dataset shape: (2506, 33)
Meal values: <ArrowStringArray>
['lunch', 'breakfast', 'closed', 'dinner']
Length: 4, dtype: str
After filtering to breakfast/lunch/dinner: (1813, 33)


,meal,waste_kg,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg,foot_traffic,is_holiday,has_special_event,year,month,...,lag_28,rolling_mean_7,rolling_mean_14,rolling_std_7,rolling_max_7,section_encoded,section_a,section_b,section_c,section_d
0,lunch,5.17,2.74,1.75,0.68,102.00,0,1,2025,1,...,0.18,2.262857,2.102857,2.249257,5.58,0,True,False,False,False
1,breakfast,4.26,1.74,1.27,1.24,46.25,0,1,2025,1,...,3.18,2.854286,2.396429,2.133197,5.58,0,True,False,False,False
3,dinner,2.72,1.72,0.75,0.25,75.00,0,1,2025,1,...,0.06,2.147143,2.390000,1.978819,5.17,0,True,False,False,False
4,lunch,6.07,3.12,2.06,0.89,138.00,0,1,2025,1,...,1.50,2.842857,2.810000,2.401317,6.07,0,True,False,False,False
5,breakfast,2.96,1.21,0.89,0.85,46.00,0,1,2025,1,...,1.82,3.244286,2.786429,2.090940,6.07,0,True,False,False,False


## Preprocess Data

- Build a proper `date` column from year/month/day.
- Sort by section, date, and meal_type.
- do lag and rolling features grouped by (section, meal_type) so each meal stream has its own temporal context.

In [40]:
df_raw['date'] = pd.to_datetime(df_raw[['year', 'month', 'day']])

sections = ['a', 'b', 'c', 'd']
def get_section_label(row):
    for s in sections:
        if row[f'section_{s}']:
            return s
    return None

df_raw['section'] = df_raw.apply(get_section_label, axis=1)

df_raw = df_raw.rename(columns={'meal': 'meal_type'})

df_raw = df_raw.sort_values(['section', 'date', 'meal_type']).reset_index(drop=True)

group_key = ['section', 'meal_type']

for lag in [1, 7, 14, 28]:
    df_raw[f'lag_{lag}'] = df_raw.groupby(group_key)[TARGET].shift(lag)

df_raw['rolling_mean_7']  = df_raw.groupby(group_key)[TARGET].transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
df_raw['rolling_mean_14'] = df_raw.groupby(group_key)[TARGET].transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
df_raw['rolling_std_7']   = df_raw.groupby(group_key)[TARGET].transform(lambda x: x.shift(1).rolling(7,  min_periods=1).std().fillna(0))
df_raw['rolling_max_7']   = df_raw.groupby(group_key)[TARGET].transform(lambda x: x.shift(1).rolling(7,  min_periods=1).max())

df_raw = df_raw.dropna(subset=[f'lag_{l}' for l in [1, 7, 14, 28]]).reset_index(drop=True)
print(f'After preprocessing: {df_raw.shape}')

section_data = {}
drop_cols = [f'section_{s}' for s in sections] + ['section_encoded', 'section']
for sec in sections:
    sec_df = df_raw[df_raw['section'] == sec].copy()
    sec_df = sec_df.drop(columns=drop_cols, errors='ignore')
    section_data[sec] = sec_df
    print(f'Section {sec.upper()}: {len(sec_df)} rows, {sec_df["meal_type"].value_counts().to_dict()}')

After preprocessing: (1477, 35)
Section A: 371 rows, {'dinner': 127, 'breakfast': 125, 'lunch': 119}
Section B: 372 rows, {'dinner': 130, 'breakfast': 125, 'lunch': 117}
Section C: 372 rows, {'dinner': 131, 'breakfast': 129, 'lunch': 112}
Section D: 362 rows, {'dinner': 131, 'breakfast': 124, 'lunch': 107}


## Define Features and Target

`meal_type` is kept as a categorical feature. `date` is dropped before training.

In [41]:
def get_X_y_cleaned(data):
    """Drop non-feature columns and encode categoricals for XGBoost."""
    X = data.drop(columns=[TARGET, 'date'], errors='ignore')
    y = data[TARGET]
    for col in X.select_dtypes(include=['object', 'string']).columns:
        X[col] = X[col].astype('category')
    return X, y

X_a, y_a = get_X_y_cleaned(section_data['a'])
print(f'Features shape: {X_a.shape}')
print('Feature columns:', list(X_a.columns))

Features shape: (371, 27)
Feature columns: ['meal_type', 'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg', 'foot_traffic', 'is_holiday', 'has_special_event', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'rolling_max_7']


## Time-Based Train-Test Split

We reserve the last 30 days (so 90 rows just 3 meals × 30 days) for testing.

In [42]:
TEST_DAYS = 30

splits = {}
for sec in sections:
    data = section_data[sec].sort_values('date').reset_index(drop=True)
    unique_dates = data['date'].drop_duplicates().sort_values()
    cutoff_date = unique_dates.iloc[-TEST_DAYS]  # first date of the test window
    train = data[data['date'] < cutoff_date].copy()
    test  = data[data['date'] >= cutoff_date].copy()
    splits[sec] = {'train': train, 'test': test}
    print(f'Section {sec.upper()} | Train: {len(train)} rows ({train["date"].nunique()} days) | '
          f'Test: {len(test)} rows ({test["date"].nunique()} days)')

Section A | Train: 292 rows (113 days) | Test: 79 rows (30 days)
Section B | Train: 294 rows (113 days) | Test: 78 rows (30 days)
Section C | Train: 297 rows (112 days) | Test: 75 rows (30 days)
Section D | Train: 288 rows (112 days) | Test: 74 rows (30 days)


## Baseline Model (Default XGBoost)

Train with default parameters to establish a performance reference.

In [43]:
def evaluate_model(model, X_test, y_test):
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)
    mape = np.mean(np.abs((y_test - pred) / np.where(y_test == 0, 1e-9, y_test))) * 100
    r2   = r2_score(y_test, pred)
    return rmse, mae, mape, r2


def evaluate_by_meal(model, test_data):
    """Return per-meal metrics as a list of dicts."""
    rows = []
    for meal in MEALS:
        meal_df = test_data[test_data['meal_type'] == meal]
        if len(meal_df) == 0:
            continue
        X_m, y_m = get_X_y_cleaned(meal_df)
        rmse, mae, mape, r2 = evaluate_model(model, X_m, y_m)
        rows.append({'meal': meal, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2})
    return rows


baseline_results = []
for sec in sections:
    train = splits[sec]['train']
    test  = splits[sec]['test']
    X_train, y_train = get_X_y_cleaned(train)
    X_test,  y_test  = get_X_y_cleaned(test)

    bl_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=RANDOM_SEED,
        verbosity=0,
        enable_categorical=True
    )
    bl_model.fit(X_train, y_train)

    # Overall metrics
    rmse, mae, mape, r2 = evaluate_model(bl_model, X_test, y_test)
    baseline_results.append({'section': sec.upper(), 'meal': 'overall', 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2})

    # Per-meal metrics
    for row in evaluate_by_meal(bl_model, test):
        row['section'] = sec.upper()
        baseline_results.append(row)

baseline_df = pd.DataFrame(baseline_results)
print('Baseline Performance')
print(baseline_df.to_string(index=False))

Baseline Performance
section      meal     RMSE      MAE     MAPE       R2
      A   overall 0.223689 0.114864 2.892394 0.990980
      A breakfast 0.078427 0.060133 2.671874 0.996767
      A     lunch 0.303077 0.165990 3.417465 0.987089
      A    dinner 0.252349 0.132789 2.682354 0.988680
      B   overall 0.879655 0.299781 4.981065 0.944194
      B breakfast 0.125931 0.095517 4.784489 0.990388
      B     lunch 1.453138 0.502190 5.004201 0.902634
      B    dinner 0.425667 0.286914 5.128075 0.970322
      C   overall 0.335762 0.171567 3.795438 0.987191
      C breakfast 0.101547 0.084233 3.581813 0.996445
      C     lunch 0.556100 0.305854 4.100157 0.975109
      C    dinner 0.172705 0.131584 3.719568 0.993705
      D   overall 0.452837 0.182443 3.097907 0.977881
      D breakfast 0.389218 0.161363 3.482312 0.958496
      D     lunch 0.693832 0.321063 3.876489 0.962235
      D    dinner 0.110073 0.080088 2.039541 0.997452


Evaluation Metrics

Primary metric: RMSE ,also looking at MAE, MAPE, and R² overall and per meal.

## Cross-Validation and Hyperparameter Optimization

Optuna Bayesian search with TimeSeriesSplit (5 folds).
Because each day has multiple rows (one per meal) TimeSeriesSplit splits on row index which naturally preserves temporal order within each section.
Up to 50 trials per section.

In [44]:
def objective(trial, X_train, y_train, cv_splits=5):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'objective':         'reg:squarederror',
        'random_state':      RANDOM_SEED,
        'verbosity':         0,
        'enable_categorical': True,
    }

    # TimeSeriesSplit respects row order; multi-meal rows are already sorted by date
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    rmse_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr,  X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr,  y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        pred = model.predict(X_val)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, pred)))

    return np.mean(rmse_scores)


N_TRIALS = 50
best_models = {}
study_logs  = {}

for sec in sections:
    print(f"\nOptimising Section {sec.upper()}\n")
    train = splits[sec]['train']
    X_train, y_train = get_X_y_cleaned(train)

    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=RANDOM_SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )

    def objective_wrapper(trial):
        return objective(trial, X_train, y_train)

    study.optimize(objective_wrapper, n_trials=N_TRIALS, show_progress_bar=True)
    study_logs[sec] = study.trials_dataframe()

    best_params = study.best_params
    print(f'Best parameters for section {sec.upper()}: {best_params}')

    # Train final model on full training set
    final_model = xgb.XGBRegressor(
        **best_params,
        objective='reg:squarederror',
        random_state=RANDOM_SEED,
        verbosity=0,
        enable_categorical=True
    )
    final_model.fit(X_train, y_train)
    best_models[sec] = final_model

    # Save model and params
    with open(f'models/xgboost_optimized/tuned_section_{sec}.pkl', 'wb') as f:
        pickle.dump(final_model, f)

    # Quick test-set evaluation
    test = splits[sec]['test']
    X_test, y_test = get_X_y_cleaned(test)
    rmse, mae, mape, r2 = evaluate_model(final_model, X_test, y_test)
    print(f'Test set performance - RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%, R2: {r2:.4f}')

print('\nOptimisation completed for all sections.')


Optimising Section A



Best trial: 42. Best value: 0.359843: 100%|██████████| 50/50 [02:58<00:00,  3.58s/it]


Best parameters for section A: {'n_estimators': 950, 'max_depth': 9, 'learning_rate': 0.056208111738860346, 'subsample': 0.6454293548907892, 'colsample_bytree': 0.9183945066241215, 'reg_lambda': 0.5597246284055408, 'reg_alpha': 0.001611631943664484, 'min_child_weight': 2}
Test set performance - RMSE: 0.1858, MAE: 0.0944, MAPE: 2.24%, R2: 0.9938

Optimising Section B



Best trial: 49. Best value: 0.196427: 100%|██████████| 50/50 [02:58<00:00,  3.57s/it]


Best parameters for section B: {'n_estimators': 700, 'max_depth': 9, 'learning_rate': 0.010309754938756096, 'subsample': 0.6252506348914477, 'colsample_bytree': 0.8227388956734512, 'reg_lambda': 0.004505241716510253, 'reg_alpha': 0.01086031327227575, 'min_child_weight': 1}
Test set performance - RMSE: 0.8524, MAE: 0.2226, MAPE: 3.16%, R2: 0.9476

Optimising Section C



Best trial: 35. Best value: 0.348078: 100%|██████████| 50/50 [02:41<00:00,  3.23s/it]


Best parameters for section C: {'n_estimators': 650, 'max_depth': 6, 'learning_rate': 0.04104437222141519, 'subsample': 0.699244765865553, 'colsample_bytree': 0.8594777475576882, 'reg_lambda': 0.007992479104126211, 'reg_alpha': 0.3234442144206603, 'min_child_weight': 1}
Test set performance - RMSE: 0.2839, MAE: 0.1382, MAPE: 2.90%, R2: 0.9908

Optimising Section D



Best trial: 23. Best value: 0.285254: 100%|██████████| 50/50 [02:58<00:00,  3.57s/it]


Best parameters for section D: {'n_estimators': 900, 'max_depth': 4, 'learning_rate': 0.022822527338608036, 'subsample': 0.8615888857369538, 'colsample_bytree': 0.9861491002170512, 'reg_lambda': 4.109769498728061, 'reg_alpha': 0.003637331048458448, 'min_child_weight': 1}
Test set performance - RMSE: 0.4192, MAE: 0.1546, MAPE: 2.66%, R2: 0.9810

Optimisation completed for all sections.


## Final Performance Comparison

Compare baseline vs tuned models overall and broken down by meal type.

In [45]:
final_results = []

for sec in sections:
    test = splits[sec]['test']
    X_test, y_test = get_X_y_cleaned(test)

    b_rmse, b_mae, b_mape, b_r2 = evaluate_model(bl_model, X_test, y_test)

    tuned_model = best_models[sec]
    t_rmse, t_mae, t_mape, t_r2 = evaluate_model(tuned_model, X_test, y_test)

    for model_name, rmse, mae, mape, r2 in [
        ('Baseline',       b_rmse, b_mae, b_mape, b_r2),
        ('Tuned (Optuna)', t_rmse, t_mae, t_mape, t_r2),
    ]:
        final_results.append({'section': sec.upper(), 'model': model_name, 'meal': 'overall',
                               'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2})

    for row in evaluate_by_meal(tuned_model, test):
        row.update({'section': sec.upper(), 'model': 'Tuned (Optuna)'})
        final_results.append(row)

results_df = pd.DataFrame(final_results)
print('Final Performance Comparison')
print(results_df.to_string(index=False))

results_df.to_csv('metrics/xgboost_model_comparison.csv', index=False)
print('\nComparison saved to metrics/xgboost_model_comparison.csv')

Final Performance Comparison
section          model      meal     RMSE      MAE     MAPE       R2
      A       Baseline   overall 0.211790 0.128525 3.494361 0.991914
      A Tuned (Optuna)   overall 0.185764 0.094439 2.240109 0.993779
      A Tuned (Optuna) breakfast 0.064999 0.047198 2.042240 0.997779
      A Tuned (Optuna)     lunch 0.283575 0.149105 2.703289 0.988697
      A Tuned (Optuna)    dinner 0.169827 0.100589 2.058683 0.994873
      B       Baseline   overall 1.132154 0.305672 4.025995 0.907559
      B Tuned (Optuna)   overall 0.852400 0.222610 3.158802 0.947599
      B Tuned (Optuna) breakfast 0.062962 0.041761 2.146040 0.997597
      B Tuned (Optuna)     lunch 1.452304 0.462598 4.572115 0.902745
      B Tuned (Optuna)    dinner 0.249264 0.154779 2.714522 0.989823
      C       Baseline   overall 0.564720 0.198531 3.196891 0.963766
      C Tuned (Optuna)   overall 0.283932 0.138170 2.895680 0.990840
      C Tuned (Optuna) breakfast 0.083027 0.067969 2.740468 0.997624
     

## 7-Day Forecast (Breakfast, Lunch, Dinner)

For each section we generate a 7-day forecast per meal type

In [46]:
def forecast_next_week_iterative(model, history_df, horizon=7, target_col='waste_kg'):
    MAX_LOOKBACK = 28  # matches lag_28
    forecasts = {meal: [] for meal in MEALS}
    history_df = history_df.sort_values(['date', 'meal_type']).reset_index(drop=True)
    last_date  = history_df['date'].max()

    meal_histories = {
        meal: history_df[history_df['meal_type'] == meal].copy()
        for meal in MEALS
    }

    for i in range(1, horizon + 1):
        next_date = last_date + pd.Timedelta(days=i)
        for meal in MEALS:
            meal_hist = meal_histories[meal]

            new_row = meal_hist.iloc[-1:].copy()
            new_row['date']        = next_date
            new_row[target_col]    = np.nan
            new_row['meal_type']   = meal

            new_row['year']        = next_date.year
            new_row['month']       = next_date.month
            new_row['day']         = next_date.day
            new_row['day_of_week'] = next_date.dayofweek
            new_row['day_of_year'] = next_date.dayofyear
            new_row['week_of_year']= int(next_date.isocalendar().week)
            new_row['quarter']     = next_date.quarter
            new_row['is_weekend']  = int(next_date.dayofweek >= 5)
            new_row['dow_sin']     = np.sin(2 * np.pi * next_date.dayofweek / 7)
            new_row['dow_cos']     = np.cos(2 * np.pi * next_date.dayofweek / 7)
            new_row['doy_sin']     = np.sin(2 * np.pi * next_date.dayofyear / 365)
            new_row['doy_cos']     = np.cos(2 * np.pi * next_date.dayofyear / 365)

            temp = pd.concat(
                [meal_hist.iloc[-MAX_LOOKBACK:], new_row],
                ignore_index=True
            )

            for lag in [1, 7, 14, 28]:
                temp[f'lag_{lag}'] = temp[target_col].shift(lag)
            temp['rolling_mean_7']  = temp[target_col].shift(1).rolling(7,  min_periods=1).mean()
            temp['rolling_mean_14'] = temp[target_col].shift(1).rolling(14, min_periods=1).mean()
            temp['rolling_std_7']   = temp[target_col].shift(1).rolling(7,  min_periods=1).std().fillna(0)
            temp['rolling_max_7']   = temp[target_col].shift(1).rolling(7,  min_periods=1).max()

            X_pred = temp.iloc[-1:].drop(columns=[target_col, 'date'], errors='ignore')
            for col in X_pred.select_dtypes(include=['object', 'string']).columns:
                X_pred[col] = X_pred[col].astype('category')

            predicted = float(model.predict(X_pred)[0])
            forecasts[meal].append(predicted)
            new_row[target_col] = predicted
            meal_histories[meal] = pd.concat([meal_hist, new_row], ignore_index=True)

    return forecasts

for sec in sections:
    with open(f'models/xgboost_optimized/tuned_section_{sec}.pkl', 'rb') as f:
        model = pickle.load(f)

    train_history = splits[sec]['train']
    forecasts = forecast_next_week_iterative(model, train_history, horizon=7, target_col=TARGET)

    print(f'\nSection {sec.upper()} 7-day forecast:')
    for meal, preds in forecasts.items():
        print(f'  {meal:10s}: {[round(p, 2) for p in preds]}')


Section A 7-day forecast:
  breakfast : [1.38, 1.38, 1.39, 1.4, 1.39, 1.4, 1.38]
  lunch     : [5.62, 5.64, 5.65, 5.63, 5.63, 5.62, 5.62]
  dinner    : [1.64, 1.62, 1.64, 1.62, 1.65, 1.64, 1.64]

Section B 7-day forecast:
  breakfast : [2.56, 2.57, 2.57, 2.58, 2.58, 2.57, 2.58]
  lunch     : [6.05, 6.0, 5.94, 5.96, 5.93, 5.93, 5.97]
  dinner    : [7.94, 7.85, 7.85, 7.87, 7.89, 7.86, 7.86]

Section C 7-day forecast:
  breakfast : [4.22, 4.22, 4.21, 4.2, 4.22, 4.21, 4.21]
  lunch     : [12.69, 12.61, 12.65, 12.73, 12.64, 12.74, 12.71]
  dinner    : [10.47, 10.39, 10.45, 10.49, 10.49, 10.51, 10.46]

Section D 7-day forecast:
  breakfast : [0.98, 0.98, 0.99, 1.0, 0.99, 0.98, 0.98]
  lunch     : [2.05, 2.03, 2.06, 2.06, 2.05, 2.05, 2.06]
  dinner    : [2.44, 2.44, 2.43, 2.42, 2.43, 2.44, 2.43]


## Forecast

In [48]:
all_forecasts_comparison = {}

for sec in sections:
    train_data = splits[sec]['train']
    baseline_preds = forecast_next_week_iterative(bl_model, train_data.copy(), horizon=7, target_col=TARGET)
    with open(f'models/xgboost_optimized/tuned_section_{sec}.pkl', 'rb') as f:
        tuned_model = pickle.load(f)
    tuned_preds = forecast_next_week_iterative(tuned_model, train_data.copy(), horizon=7, target_col=TARGET)
    all_forecasts_comparison[sec.upper()] = {'Baseline': baseline_preds, 'Tuned': tuned_preds}

In [49]:
last_train_date = splits['a']['train']['date'].max()
forecast_dates  = pd.date_range(start=last_train_date + pd.Timedelta(days=1), periods=7)

plot_rows = []
for sec, model_dict in all_forecasts_comparison.items():
    for model_name, meal_dict in model_dict.items():
        for meal, preds in meal_dict.items():
            for date, val in zip(forecast_dates, preds):
                plot_rows.append({
                    'Date': date,
                    'Section': sec,
                    'Model': model_name,
                    'Meal': meal,
                    'Forecast': val
                })

forecast_df_plot = pd.DataFrame(plot_rows)
display(forecast_df_plot.head(12))

,Date,Section,Model,Meal,Forecast
0,2025-06-01,A,Baseline,breakfast,1.310285
1,2025-06-02,A,Baseline,breakfast,1.316293
2,2025-06-03,A,Baseline,breakfast,1.334215
3,2025-06-04,A,Baseline,breakfast,1.313418
4,2025-06-05,A,Baseline,breakfast,1.328460
5,2025-06-06,A,Baseline,breakfast,1.325176
6,2025-06-07,A,Baseline,breakfast,1.318583
7,2025-06-01,A,Baseline,lunch,5.455874
8,2025-06-02,A,Baseline,lunch,5.577167
9,2025-06-03,A,Baseline,lunch,5.560900
